# Quick gNATSGO STAC Fetch Test

Tests `fetch_stac_cog()` with per-variable `asset_key` for gNATSGO rasters,
without running the full pipeline. Uses a small subset of DRB HRUs.

In [ ]:
import os

# Ensure PROJ_DATA is set for the Jupyter kernel (VS Code may not inherit pixi activation)
proj_data = os.path.join(os.getcwd(), ".pixi", "envs", "dev", "share", "proj")
if os.path.isdir(proj_data):
    os.environ["PROJ_DATA"] = proj_data

import geopandas as gpd

from hydro_param.data_access import fetch_stac_cog
from hydro_param.dataset_registry import DatasetEntry


In [4]:
# Load a small subset of the DRB fabric for testing
fabric = gpd.read_file("../data/pywatershed_gis/drb_2yr/nhru.gpkg")
print(f"Full fabric: {len(fabric)} features, CRS={fabric.crs}")

# Take first 10 features for a quick test
subset = fabric.head(10)
bounds = subset.to_crs("EPSG:4326").total_bounds  # [minx, miny, maxx, maxy]
bbox = bounds.tolist()
print(f"Test bbox (WGS84): {bbox}")


Full fabric: 765 features, CRS=EPSG:5070
Test bbox (WGS84): [-75.56557464651286, 38.68388748175658, -75.13053894028573, 38.9890251161193]


In [5]:
# gNATSGO dataset entry matching configs/datasets/soils.yml
entry = DatasetEntry(
    strategy="stac_cog",
    catalog_url="https://planetarycomputer.microsoft.com/api/stac/v1",
    collection="gnatsgo-rasters",
    crs="EPSG:5070",
    sign="planetary_computer",
)

# Test each variable with its per-variable asset_key
variables = {
    "aws0_100": "Available water storage 0-100cm",
    "rootznemc": "Root zone depth",
    "rootznaws": "Root zone available water storage",
}


In [6]:
for var_name, description in variables.items():
    print(f"\n--- Fetching {var_name} ({description}) ---")
    da = fetch_stac_cog(entry, bbox, asset_key=var_name)
    print(f"  Shape: {da.shape}")
    print(f"  CRS: {da.rio.crs}")
    print(f"  Min: {float(da.min()):.4f}, Max: {float(da.max()):.4f}, Mean: {float(da.mean()):.4f}")
    print(f"  NoData count: {int(da.isnull().sum())} / {da.size}")



--- Fetching aws0_100 (Available water storage 0-100cm) ---
  Shape: (4147, 4392)
  CRS: ESRI:102039
  Min: 2.0000, Max: 402.8100, Mean: 140.9754
  NoData count: 3868926 / 18213624

--- Fetching rootznemc (Root zone depth) ---
  Shape: (4147, 4392)
  CRS: ESRI:102039
  Min: 0.0000, Max: 150.0000, Mean: 126.6811
  NoData count: 4332250 / 18213624

--- Fetching rootznaws (Root zone available water storage) ---
  Shape: (4147, 4392)
  CRS: ESRI:102039
  Min: 0.0000, Max: 600.0000, Mean: 154.5681
  NoData count: 4332250 / 18213624


In [7]:
# Verify the old default asset_key='data' would fail
try:
    fetch_stac_cog(entry, bbox)  # no asset_key override -> uses 'data'
    print("ERROR: Should have raised KeyError!")
except KeyError as e:
    print(f"Correctly raised KeyError: {e}")


Correctly raised KeyError: "Asset key 'data' not found in STAC item 'conus_1739845_2072225_1903685_1908385' (collection='gnatsgo-rasters'). Available data assets: ['aws0_100', 'aws0_150', 'aws0_20', 'aws0_30', 'aws0_5', 'aws0_999', 'aws100_150', 'aws150_999', 'aws20_50', 'aws50_100', 'aws5_20', 'droughty', 'mukey', 'musumcpct', 'musumcpcta', 'musumcpcts', 'nccpi3all', 'nccpi3corn', 'nccpi3cot', 'nccpi3sg', 'nccpi3soy', 'pctearthmc', 'pwsl1pomu', 'rootznaws', 'rootznemc', 'soc0_100', 'soc0_150', 'soc0_20', 'soc0_30', 'soc0_5', 'soc0_999', 'soc100_150', 'soc150_999', 'soc20_50', 'soc50_100', 'soc5_20', 'tk0_100a', 'tk0_100s', 'tk0_150a', 'tk0_150s', 'tk0_20a', 'tk0_20s', 'tk0_30a', 'tk0_30s', 'tk0_5a', 'tk0_5s', 'tk0_999a', 'tk0_999s', 'tk100_150a', 'tk100_150s', 'tk150_999a', 'tk150_999s', 'tk20_50a', 'tk20_50s', 'tk50_100a', 'tk50_100s', 'tk5_20a', 'tk5_20s']. Check the 'asset_key' field in your dataset registry."


In [ ]:
# Quick zonal stats test with one variable
from hydro_param.data_access import save_to_geotiff
from hydro_param.processing import ZonalProcessor
from pathlib import Path
import tempfile

da = fetch_stac_cog(entry, bbox, asset_key="aws0_100")

with tempfile.TemporaryDirectory() as tmpdir:
    tiff_path = Path(tmpdir) / "aws0_100.tif"
    save_to_geotiff(da, tiff_path)

    processor = ZonalProcessor()
    result = processor.process(
        fabric=subset,
        tiff_path=tiff_path,
        variable_name="aws0_100",
        id_field="nhm_id",
        engine="exactextract",
        statistics=["mean"],
        categorical=False,
        source_crs="EPSG:5070",
    )

print(f"\nZonal stats for aws0_100 ({len(result)} features):")
print(result.head(10))


Processing 5317: 100%|██████████| 100.0/100 [00:00<00:00, 615.50it/s]



Zonal stats for aws0_100 (10 features):
              mean
nhm_id            
5308    152.978335
5309    247.024427
5310    226.829287
5311    119.141319
5312    167.049387
5313    124.348783
5314    132.149616
5315     84.107954
5316    133.083605
5317    158.469267
